In [ ]:
# ==============================================================================
# ENVIRONMENT SETUP (Execute ONLY the first time or in a fresh virtual environment)
# ==============================================================================

# 1. Install Jupyter kernel support for PyCharm and PyTorch with CUDA 12.1 support
# This ensures the model runs on the GPU rather than the CPU.
!pip install ipykernel torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 2. Install Unsloth optimized for local Ampere/Turing GPUs (e.g., RTX 2080 Super, RTX 30xx, RTX 40xx)
# We avoid the [colab-new] tag since we are running this locally, not on Google Colab.
!pip install "unsloth[cu121-ampere-torch211] @ git+https://github.com/unslothai/unsloth.git"

# 3. Install the Hugging Face ecosystem and other required dependencies
# The --no-deps flag is crucial: it prevents pip from overwriting the specific library versions
# that Unsloth relies on, avoiding conflicts and ensuring maximum training speed.
!pip install --no-deps trl peft accelerate bitsandbytes datasets

print("Installation complete! Please restart the Jupyter kernel before proceeding to the next cells.")

In [ ]:
import torch
import gc
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig

# Deep VRAM cleanup to prevent crashes
gc.collect()
torch.cuda.empty_cache()
print("Libraries imported and GPU memory cleaned!")

In [ ]:
print("Loading dataset...")

# Use relative path (searches in the current directory)
dataset_path = "./dataset.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")

print(f"Dataset successfully loaded! Number of examples: {len(dataset)}")

In [ ]:
print("Loading Qwen model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-3B-Instruct",
    max_seq_length = 3000, # <-- Safety limit for 8GB GPUs
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_dora = False,
)
print("Model and LoRA adapters ready!")

In [ ]:
print("Formatting data in ChatML style...")
tokenizer = get_chat_template(tokenizer, chat_template = "chatml")

def format_data(examples):
    texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
    return {"text": texts}

dataset = dataset.map(format_data, batched=True)
print("Data ready for training.")

In [ ]:
print("Starting training...")

# One last memory sweep before starting
gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 3000,
    dataset_num_proc = 2,
    args = SFTConfig(
        per_device_train_batch_size = 1, # <-- Starting cautiously for the RTX 2080 Super
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 5,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        seed = 3407,
        output_dir = "./chatbdi_results", # Intermediate saves here
        save_steps = 50, # Save a checkpoint every 50 steps
    ),
)

trainer.train()
print("Training completed!")

In [ ]:
print("Saving LoRA adapters locally...")
lora_path = "./lora_chatbdi"

model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"LoRA adapters saved in: {lora_path}")

print("Exporting to GGUF. This will take a few minutes...")
export_path = "./chatbdi_model"

model.save_pretrained_gguf(
    export_path,
    tokenizer,
    quantization_method = "q4_k_m",
)
print(f"🎉 FINISHED! Your GGUF model has been safely saved in: {export_path}")